# Build the real-text annotation set

Downloads **HumAID** — 76,484 human-labelled tweets from 19 real disasters
— samples a balanced subset, and writes two CSVs for two people to label
independently.

Run with **Runtime → Run all**, then download the two CSVs at the end.

**Why this exists:** every number in the project so far comes from text the
authors or a generator wrote. That is the one thing a reviewer will not
accept. This replaces it with text real people wrote during real disasters.

**What HumAID does not give you:** its labels say what a tweet is *about*
(information type), not what the emergency *is*. There is no label for
casualty count, mobility, hazard or severity, and no class for TRAPPED,
FLOOD or FIRE. So HumAID supplies the text; the labels are still yours to
make. It is also **English only** — Hindi has to be collected separately.

License: CC BY-NC-SA 4.0. Cite the ICWSM 2021 paper and keep it
non-commercial.


In [ ]:
!pip install -q datasets
from datasets import load_dataset
import pandas as pd, numpy as np

ds = load_dataset('QCRI/HumAID-all')
df = pd.concat([ds[s].to_pandas() for s in ds.keys()], ignore_index=True)
print(f'{len(df):,} tweets')
df['class_label'].value_counts()


## Sampling

Not uniform. Classes describing an emergency in progress are worth more
than sympathy and donation appeals, which are real tweets but not the kind
of message this system carries. Some of the latter are kept anyway — the
encoder must refuse them correctly, and that refusal is part of the test.


In [ ]:
# Weight by how likely a class is to contain an actual emergency report.
WEIGHTS = {
    'injured_or_dead_people':              120,
    'requests_or_urgent_needs':            120,
    'rescue_volunteering_or_donation_effort': 60,
    'displaced_people_and_evacuations':     60,
    'infrastructure_and_utility_damage':    60,
    'missing_or_found_people':              40,
    'caution_and_advice':                   20,
    'other_relevant_information':           20,
    'sympathy_and_support':                 10,   # should be SKIPped - tests refusal
    'not_humanitarian':                     10,
    'dont_know_or_cant_judge':               5,
}

rng = np.random.default_rng(42)
parts = []
for cls, n in WEIGHTS.items():
    pool = df[df['class_label'] == cls]
    if len(pool) == 0:
        print(f'  (no rows for {cls})'); continue
    parts.append(pool.sample(min(n, len(pool)), random_state=42))

sample = pd.concat(parts).sample(frac=1, random_state=7).reset_index(drop=True)
sample = sample.drop_duplicates(subset='tweet_text').reset_index(drop=True)
print(f'sampled {len(sample)} tweets')
sample['class_label'].value_counts()


## A suggested category, which is only a suggestion

Four HumAID classes map onto a category cleanly. The rest do not, and the
ones that matter most — TRAPPED, FLOOD, FIRE — have no HumAID class at all.

The suggestion is pre-filled to save typing. **Annotators must read the text
and overwrite it where it is wrong**, which will be often. Accepting the
suggestion unread would just relabel HumAID's taxonomy under new names.


In [ ]:
SUGGEST = {
    'injured_or_dead_people':                 'MEDICAL',
    'requests_or_urgent_needs':               'SUPPLIES_NEEDED',
    'displaced_people_and_evacuations':       'EVACUATION',
    'infrastructure_and_utility_damage':      'COLLAPSE',
    'rescue_volunteering_or_donation_effort': 'RESCUE_NEEDED',
}

out = pd.DataFrame({
    'id':                range(1, len(sample) + 1),
    'tweet_text':        sample['tweet_text'],
    'humaid_label':      sample['class_label'],
    'suggested_category': sample['class_label'].map(SUGGEST).fillna(''),
    'category':          '',
    'casualty_count':    '',
    'mobility':          '',
    'hazard_flags':      '',
    'severity':          '',
    'notes':             '',
})
print(f"{(out['suggested_category'] != '').mean():.0%} of rows got a suggestion; "
      f'the rest start blank')
out.head(8)


## Write the two sheets


In [ ]:
for who in ('A', 'B'):
    out.to_csv(f'annotator_{who}.csv', index=False)

legend = pd.DataFrame({'field': ['category','casualty_count','mobility','hazard_flags','severity'],
 'allowed_values': [
   'TRAPPED COLLAPSE MEDICAL FIRE FLOOD EVACUATION SUPPLIES_NEEDED RESCUE_NEEDED OTHER',
   '0-7  (0 = not stated, 7 = seven or more)',
   'UNKNOWN AMBULATORY TRAPPED UNRESPONSIVE',
   'FIRE WATER GAS STRUCTURAL ELECTRICAL CHEMICAL  (comma-separated, blank = none)',
   'UNKNOWN NONE MINOR SERIOUS']})
legend.to_csv('legend.csv', index=False)

from google.colab import files
for f in ('annotator_A.csv', 'annotator_B.csv', 'legend.csv'):
    files.download(f)
print('downloading 3 files')


## Next

1. Put both CSVs in `data/annotated/` in the repo.
2. Two people label **independently**, following `docs/ANNOTATION_GUIDE.md`.
   Not together — agreement measured after discussion is meaningless.
3. Score it:

   ```bash
   python nlp/score_annotations.py data/annotated/annotator_A.csv data/annotated/annotator_B.csv
   ```

   It prints Cohen's kappa per field, writes the disagreements to work
   through, and then re-runs the encoder comparison on real text.

Budget roughly 6–8 hours per person for ~500 tweets. That is the slow part,
and it is the part that makes every other number in the paper mean something.

**Hindi is not in here.** HumAID is English only, so the code-mixed claim
needs its own collection — scenarios given to 25–30 people, their own words.
Keep it separate and describe it as elicited, not in-the-wild.
